# Solving Linear Systems

## Definition
A **linear system** is a set of equations of the form `Ax = b`, where:
- `A` is a known **coefficient matrix** (n × n for square systems)
- `b` is a known **right-hand side vector**
- `x` is the **unknown vector** to solve for

### Methods
| Method | Function | When to Use |
|--------|----------|-------------|
| Direct solve | `np.linalg.solve(A, b)` | Square A, prefer over inv |
| Inverse multiply | `np.linalg.inv(A) @ b` | Rarely — numerically unstable |
| Least squares | `np.linalg.lstsq(A, b)` | Over/underdetermined systems |
| Pseudo-inverse | `np.linalg.pinv(A) @ b` | Singular or non-square A |
| Cholesky | `np.linalg.cholesky(A)` | A is symmetric positive definite |

### AI/ML Connections
- **Normal equations** for linear regression: `(X^T X) θ = X^T y`
- **Gaussian processes**: solve K α = y
- **Ridge regression**: `(X^T X + λI) θ = X^T y`

In [ ]:
import numpy as np

# Solve 2x2 system:
# 2x + y  = 5
# x  + 3y = 10
A = np.array([[2, 1],
              [1, 3]], dtype=float)
b = np.array([5, 10], dtype=float)

x = np.linalg.solve(A, b)
print('Solution x:', x)
print('Verify Ax = b:', np.allclose(A @ x, b))

In [ ]:
# Solving a 3x3 system
A = np.array([[2, -1,  0],
              [-1, 2, -1],
              [0, -1,  2]], dtype=float)
b = np.array([1, 0, 1], dtype=float)

x = np.linalg.solve(A, b)
print('Solution:', x)
print('Residual ||Ax - b||:', np.linalg.norm(A @ x - b))

In [ ]:
# Why solve() is better than inv() @ b
import time
rng = np.random.default_rng(42)
n = 500
A = rng.standard_normal((n, n))
b = rng.standard_normal(n)

# Method 1: solve (uses LU decomposition, numerically stable)
t0 = time.time()
x1 = np.linalg.solve(A, b)
t1 = time.time()
print(f'solve(): {t1-t0:.4f}s  residual={np.linalg.norm(A@x1-b):.2e}')

# Method 2: explicit inverse (avoid this!)
t0 = time.time()
x2 = np.linalg.inv(A) @ b
t1 = time.time()
print(f'inv()@b: {t1-t0:.4f}s  residual={np.linalg.norm(A@x2-b):.2e}')

In [ ]:
# Solving multiple right-hand sides at once
A = np.array([[2, 1],
              [1, 3]], dtype=float)

# b is now a matrix (each column is a different rhs)
B = np.array([[5, 3],
              [10, 2]], dtype=float)

X = np.linalg.solve(A, B)
print('Solutions X:\n', X)
print('Verify AX = B:\n', np.allclose(A @ X, B))

In [ ]:
# Least squares: overdetermined system (more equations than unknowns)
# Linear regression: fit y = ax + b to 4 data points
x_data = np.array([1, 2, 3, 4], dtype=float)
y_data = np.array([2, 4, 5, 4], dtype=float)

# Build design matrix A (n x 2): [x, 1]
A = np.column_stack([x_data, np.ones_like(x_data)])
print('Design matrix A:\n', A)

# lstsq returns (solution, residuals, rank, singular_values)
result = np.linalg.lstsq(A, y_data, rcond=None)
theta = result[0]
print(f'Slope: {theta[0]:.4f}, Intercept: {theta[1]:.4f}')
print('Predictions:', A @ theta)
print('Residuals:', y_data - A @ theta)

In [ ]:
# Ridge regression: (X^T X + lambda * I) theta = X^T y
rng = np.random.default_rng(42)
n_samples, n_features = 100, 20
X = rng.standard_normal((n_samples, n_features))
y = rng.standard_normal(n_samples)

lam = 0.1  # regularization strength

# Normal equations with ridge penalty
A = X.T @ X + lam * np.eye(n_features)
b = X.T @ y
theta_ridge = np.linalg.solve(A, b)
print('Ridge theta shape:', theta_ridge.shape)
print('Ridge theta norm:', np.round(np.linalg.norm(theta_ridge), 4))

In [ ]:
# EXCEPTION: Singular matrix — solve fails
A_singular = np.array([[1, 2], [2, 4]])  # row2 = 2*row1
b = np.array([3, 6])
try:
    np.linalg.solve(A_singular, b)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)

# Use lstsq instead for degenerate systems
result = np.linalg.lstsq(A_singular, b, rcond=None)
print('lstsq solution:', result[0])